# Round 2: Tree-based Models (Random Forest & XGBoost)

**Goal**: Implement non-linear baselines.
- **Random Forest**: Reduced feature space.
- **XGBoost**: Dimensionality reduction via TruncatedSVD.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set(style='whitegrid')
reports_dir = 'round2/reports'
os.makedirs(reports_dir, exist_ok=True)
print("Libraries imported.")

## 1. Load Data

In [ ]:
train_df = pd.read_excel('round2/train.xlsx')
val_df = pd.read_excel('round2/val.xlsx')

X_train = train_df['cleaned_poem'].astype(str)
X_val = val_df['cleaned_poem'].astype(str)

# Maps
with open('round2/label_maps.json', 'r') as f:
    maps = json.load(f)
p_map = {v: k for k, v in maps['primary_map'].items()}
s_map = {v: k for k, v in maps['secondary_map'].items()}

## 2. Train Helper for Random Forest

In [ ]:
def train_rf(y_train, y_val, label_map, prefix):
    print(f"\nTraining Random Forest for {prefix}...")
    # Pipeline: TF-IDF (top 5000) -> RF
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=5000)),
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    macro = f1_score(y_val, y_pred, average='macro')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro:.4f}")
    
    # Save Metrics
    metrics = {'accuracy': acc, 'macro_f1': macro}
    with open(f'{reports_dir}/{prefix}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=4)


## 3. Train Helper for XGBoost

In [ ]:
def train_xgb(y_train, y_val, prefix):
    print(f"\nTraining XGBoost for {prefix}...")
    # Pipeline: TF-IDF -> SVD -> XGB
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=5)),
        ('svd', TruncatedSVD(n_components=100, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    
    # XGB requires 0-indexed encoded labels. Ensure input is correct.
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    macro = f1_score(y_val, y_pred, average='macro')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro:.4f}")
    
    metrics = {'accuracy': acc, 'macro_f1': macro}
    with open(f'{reports_dir}/{prefix}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=4)

## 4. Run Tree Models

In [ ]:
y_train_p = train_df['primary_id']
y_val_p = val_df['primary_id']
y_train_s = train_df['secondary_id']
y_val_s = val_df['secondary_id']

# Random Forest
train_rf(y_train_p, y_val_p, p_map, 'rf_primary')
train_rf(y_train_s, y_val_s, s_map, 'rf_secondary')

# XGBoost
train_xgb(y_train_p, y_val_p, 'xgb_primary')
train_xgb(y_train_s, y_val_s, 'xgb_secondary')